In [8]:
import requests
import json
import pandas as pd
from io import StringIO
import time
import random
import logging


In [9]:
url = "http://www.ebi.ac.uk/ols4/api/ontologies/efo/children?id=EFO:0000319"
headers = {
        "Content-Type": "application/json",
    }
response = requests.post(url, headers=headers, verify=True)
if response.status_code == 200:
    resp_txt = response.text
    data = json.loads(resp_txt)

In [16]:
data

{'_embedded': {'terms': [{'iri': 'http://www.ebi.ac.uk/efo/EFO_0004264',
    'lang': 'en',
    'description': ['A general term used to describe any disease affecting blood vessels]. It includes vascular abnormalities caused by degenerative, metabolic and inflammatory conditions, embolic diseases, coagulative disorders, and functional disorders such as posteri or reversible encephalopathy syndrome.',
     'Pathological processes involving any of the BLOOD VESSELS in the cardiac or peripheral circulation. They include diseases of ARTERIES; VEINS; and rest of the vasculature system in the body.',
     'The etiology of vasculopathy is generally unknown and the condition is frequently not pathologically proven. Vasculitis, on the other hand, is a more specific term and is defined as inflammation of the wall of a blood vessel. However, the term vasculopathy is also used for “vasculitis” that has not been pathologically established.'],
    'synonyms': ['disease of vasculature',
     'disease 

In [103]:
data['_embedded']['terms'][1]['annotation']['gwas_trait']
data['_embedded']['terms'][1]['has_children']

False

In [53]:
id9 = [r for r in data['_embedded']['terms'][0]['annotation'].get('has_dbxref', []) if "ICD9" in r] if 'annotation' in data['_embedded']['terms'][0] else []
id10 = [r for r in data['_embedded']['terms'][0]['annotation'].get('has_dbxref', []) if "ICD10" in r] if 'annotation' in data['_embedded']['terms'][0] else []

In [55]:
data['_embedded']['terms'][0].get('term_replaced_by', '')

In [4]:
terms = [data['_embedded']['terms'][1]['label']] + data['_embedded']['terms'][1]['synonyms']
if bool(data['_embedded']['terms'][1]['term_replaced_by']):
    terms.append(data['_embedded']['terms'][1]['term_replaced_by'])
oboID = data['_embedded']['terms'][1]['obo_id']


In [37]:
data['_embedded']['terms']

[{'iri': 'http://purl.obolibrary.org/obo/MONDO_0007318',
  'lang': 'en',
  'description': ['Alagille (AGS) syndrome is variably characterized by chronic cholestasis due to paucity of intrahepatic bile ducts, peripheral pulmonary artery stenosis, vertebrae segmentation anomalies, characteristic facies, posterior embryotoxon/anterior segment abnormalities, pigmentary retinopathy, and dysplastic kidneys.'],
  'synonyms': ['Alagille syndrome',
   'Alagille-Watson syndrome',
   'Arteriohepatic dysplasia',
   'syndromic bile duct paucity',
   'Cardiovertebral syndrome',
   'Hepatofacioneurocardiovertebral syndrome',
   'Watson Alagille syndrome',
   'Watson-Miller syndrome',
   'hepatic ductular hypoplasia',
   'paucity of interlobular bile ducts'],
  'annotation': {'closeMatch': ['http://identifiers.org/meddra/10053870'],
   'exactMatch': ['http://identifiers.org/mesh/D016738',
    'http://identifiers.org/snomedct/31742004',
    'http://linkedlifedata.com/resource/umls/id/C0085280',
    'ht

In [39]:
url = f"http://www.ebi.ac.uk/ols4/api/ontologies/efo/children?id={oboID}&size=30&page=0"
headers = {
        "Content-Type": "application/json",
    }
response = requests.post(url, headers=headers, verify=True)
if response.status_code == 200:
    resp_txt = response.text
    data = json.loads(resp_txt)

In [1]:
import json

data = json.load(open("/group/glastonbury/soumick/downloads/OLS/ontology_tree_0000319.json", "r"))
data_old = json.load(open("/group/glastonbury/soumick/downloads/OLS/orig_ontology_tree_0000319.json", "r"))

In [24]:
from deepdiff import DeepDiff
diff = DeepDiff(data[0]['children'], data_old, ignore_order=True)

5

In [27]:
data[0].keys()

dict_keys(['oboID', 'terms', 'ICD9', 'ICD10', 'children'])

In [17]:
url = "http://www.ebi.ac.uk/ols4/api/ontologies/efo/parents?id=EFO:0000319"
headers = {
        "Content-Type": "application/json",
    }
response = requests.post(url, headers=headers, verify=True)
if response.status_code == 200:
    resp_txt = response.text
    data = json.loads(resp_txt)

In [25]:
term = data['_embedded']['terms'][0]
oboID = term.get('obo_id', '')

# if oboID.startswith('EFO:'):
id9 = [r for r in term['annotation'].get('has_dbxref', []) if "ICD9" in r] if 'annotation' in term else []
id10 = [r for r in term['annotation'].get('has_dbxref', []) if "ICD10" in r] if 'annotation' in term else []
terms = [term.get('label', '')] + term.get('synonyms', [])
if bool(term.get('term_replaced_by', '')):
    terms.append(term['term_replaced_by'])
oboID = term.get('obo_id', '')

node = {'oboID': oboID, 'terms': terms, 'ICD9': id9, 'ICD10': id10, 'children': []}

In [20]:
from pandasgwas.get_traits import get_traits
from pandasgwas import get_child_efo
from pandasgwas.get_studies import get_studies

import pandas as pd

In [42]:
studies = get_studies(efo_id="EFO:0007986".replace("EFO:", "EFO_"))

In [44]:
terms = list(studies.studies['diseaseTrait.trait'].unique())
print(list({t.lower() for t in terms}))

['reticulocyte count (ukb data field 30250) (cnv deletion-only model)', 'immature reticulocytes/reticulocytes.total in blood', 'reticulocytes/100 erythrocytes in blood by automated count', 'immature fraction of reticulocytes', 'reticulocyte count (ukb data field 30250) (gene-based burden)', 'reticulocyte count (ukb data field 30250)', 'high light scatter reticulocyte count (ukb data field 30300) (gene-based burden)', 'high light scatter reticulocyte count', 'reticulocyte fraction of red cells', 'high light scatter reticulocyte percentage of red cells', 'reticulocyte count (ukb data field 30250) (cnv mirror model)', 'high light scatter reticulocyte count (ukb data field 30300)', 'reticulocyte count (ukb data field 30250) (cnv duplication-only model)', 'reticulocyte count']


In [1]:
import pandas as pd
df = pd.read_csv("/project/ukbblatent/Out/Results/F20208_heart_1Ses_time2slc_MskCrop128_V2_3D100ep_L1_4ChTrans128fold0_prec32_pythaemodel-rhvae/GWAS_fullDS/Qntl_WBRIT_time2slc_Msk_V2_3D100ep_L1_128RHVAE_FP32_fullDS/results/analyses/LDLink_LDtraits/all.tagged.ldtrait.tsv", sep="\t")

In [2]:
df[df.tags.isna()]

,Query,GWAS Trait,RS Number,Position (GRCh37),Alleles,R2,D',Risk Allele,Effect Size (95% CI),Beta or OR,P-value,Pheno,SNP,tags
22,rs917727,Edge-level brain connectivity (multivariate an...,rs10234624,chr7:120959135,"C=0.65, T=0.35",0.447754,0.833705,0.632419,6.065723,NaN,1.000000e-09,Z27,rs917727_C_T,NaN
45,rs917727,Medication use (drugs affecting bone structure...,rs3779381,chr7:120966790,"A=0.744, G=0.256",0.913458,0.958241,0.258819,0.113870,0.078-0.15,5.000000e-10,Z27,rs917727_C_T,NaN
46,rs917727,Medication use (drugs affecting bone structure...,rs3779381,chr7:120966790,"A=0.744, G=0.256",0.913458,0.958241,NR,0.102400,0.072-0.133,5.000000e-11,Z27,rs917727_C_T,NaN
74,rs917727,QC T1-to-standard nonlinear alignment discrepancy,rs7808155,chr7:121000931,"A=0.74, G=0.26",0.984580,1.000000,0.279,0.122000,0.1-0.14,6.000000e-31,Z27,rs917727_C_T,NaN
110,rs7793554,Edge-level brain connectivity (multivariate an...,rs10234624,chr7:120959135,"C=0.65, T=0.35",0.117892,0.420200,0.632419,6.065723,NaN,1.000000e-09,Z27,rs7793554_A_G,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4970,rs9740264,Neurociticism,rs4444227,chr13:59116160,"C=0.773, T=0.227",0.194352,0.556199,0.214472,6.040000,NaN,2.000000e-09,Z33,rs9740264_C_A,NaN
4971,rs9740264,Feeling nervous,rs11619066,chr13:59141522,"C=0.221, G=0.779",0.188612,0.557416,0.215297,5.690000,NaN,1.000000e-08,Z33,rs9740264_C_A,NaN
4972,rs9740264,Neuroticism,rs1330745,chr13:59159949,"A=0.788, C=0.212",0.203026,0.593794,NR,NaN,NaN,4.000000e-08,Z33,rs9740264_C_A,NaN
4973,rs9740264,Worry,rs1330745,chr13:59159949,"A=0.788, C=0.212",0.203026,0.593794,NR,0.018190,0.013-0.024,3.000000e-10,Z33,rs9740264_C_A,NaN
